In [0]:
# Databricks notebook source
import sys
import os
from pyspark.sql.functions import lit

# --- Parameter Retrieval (Environment Layer) ---
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("raw_path", "") 
dbutils.widgets.text("job_run_id", "") 

CATALOG = dbutils.widgets.get("catalog_name") 
RAW_PATH = dbutils.widgets.get("raw_path")
JOB_RUN_ID = dbutils.widgets.get("job_run_id")

if not CATALOG or not RAW_PATH:
    raise ValueError("CATALOG or RAW_PATH parameter is strictly required by the parent job.")

# --- Dynamic Root Discovery (Pathing Layer) ---
project_root = os.getcwd()
while project_root != '/' and not os.path.exists(os.path.join(project_root, 'src')):
    project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- Explicit Library Imports (Modularity Layer) ---
from src.config.paths import ProjectConfig
from src.config.schemas import repayments_schema
from src.utils.spark_io import read_autoloader, write_append_stream
from src.utils.audit import get_batch_id, add_bronze_metadata

# Instantiate Configuration
cfg = ProjectConfig(catalog=CATALOG, raw_path=RAW_PATH)

In [0]:
# --- Pipeline Execution (Logic Layer) ---
batch_id = get_batch_id(JOB_RUN_ID)
print(f"Executing Customer Ingestion \nCatalog: {cfg.CATALOG} \nBatch ID: {batch_id}")

# 1. LANDING -> BRONZE (Auto Loader)
df_raw = read_autoloader(
    spark=spark, 
    source_path=cfg.SRC_REPAYMENTS, 
    schema=repayments_schema
).withColumn("_batch_id", lit(batch_id))

# 2. AUDIT METADATA
df_final = add_bronze_metadata(df_raw)

# 3. STREAM SINK (Delta Append)
query = write_append_stream(
    df=df_final, 
    target_table=cfg.REPAYMENTS_BRONZE, 
    checkpoint_loc=cfg.BRONZE_CHECKPOINT_REPAYMENTS
)

# Block notebook termination until micro-batch is committed
query.awaitTermination() 

In [0]:
# %sql
# -- select count(1) from lending_risk.bronze.repayments;
# select * from lending_risk_dev.bronze.repayments limit 3;